<a href="https://colab.research.google.com/github/peperjet/deep-learning/blob/main/Subword_Tokenizer/13_2_sentencepiece.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

센텐스피스(SentencePiece)
- 논문 : https://arxiv.org/pdf/1808.06226.pdf
- 센텐스피스 깃허브 : https://github.com/google/sentencepiece

IMDB 리뷰 토큰화하기

In [1]:
import sentencepiece as spm
import pandas as pd
import urllib.request
import csv


IMDB 리뷰 데이터 다운로드

In [2]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/LawrenceDuan/IMDb-Review-Analysis/master/IMDb_Reviews.csv", filename="IMDb_Reviews.csv")


('IMDb_Reviews.csv', <http.client.HTTPMessage at 0x7b2629387380>)

In [8]:
train_df = pd.read_csv('/content/IMDb_Reviews.csv')
train_df['review']


,review
0,My family and I normally do not watch local mo...
1,"Believe it or not, this was at one time the wo..."
2,"After some internet surfing, I found the ""Home..."
3,One of the most unheralded great works of anim...
4,"It was the Sixties, and anyone with long hair ..."
...,...
49995,the people who came up with this are SICK AND ...
49996,"The script is so so laughable... this in turn,..."
49997,"""So there's this bride, you see, and she gets ..."
49998,Your mind will not be satisfied by this nobud...


In [9]:
print('리뷰 개수 :',len(train_df)) # 리뷰 개수 출력


리뷰 개수 : 50000


센텐스피스의 입력으로 사용하기 위해서 데이터프레임을 txt 파일로 저장합니다.

In [10]:
with open('imdb_review.txt', 'w', encoding='utf8') as f:
    f.write('\n'.join(train_df['review']))


In [14]:
spm.SentencePieceTrainer.Train('--input=imdb_review.txt --model_prefix=imdb --vocab_size=5000 --model_type=bpe --max_sentence_length=9999')


input : 학습시킬 파일
model_prefix : 만들어질 모델 이름
vocab_size : 단어 집합의 크기
model_type : 사용할 모델 (unigram(default), bpe, char, word)
max_sentence_length: 문장의 최대 길이
pad_id, pad_piece: pad token id, 값
unk_id, unk_piece: unknown token id, 값
bos_id, bos_piece: begin of sentence token id, 값
eos_id, eos_piece: end of sequence token id, 값
user_defined_symbols: 사용자 정의 토큰
vocab 생성이 완료되면 imdb.model, imdb.vocab 파일 두개가 생성 됩니다. vocab 파일에서 학습된 서브워드들을 확인할 수 있습니다. 단어 집합의 크기를 확인하기 위해 vocab 파일을 데이터프레임에 저장해봅시다.

In [15]:
vocab_list = pd.read_csv('imdb.vocab', sep='\t', header=None, quoting=csv.QUOTE_NONE)
vocab_list.sample(10)


,0,1
2898,▁af,-2895
4979,z,-4976
2024,▁knew,-2021
282,▁bec,-279
1877,▁200,-1874
465,ving,-462
3105,appe,-3102
2657,▁Che,-2654
3069,▁compet,-3066
3892,▁lover,-3889


In [16]:
len(vocab_list)


5000

vocab_size의 인자를 통해 단어 집합의 크기를 5,000개로 제한하였으므로 단어 집합의 크기는 5,000개입니다.

model 파일을 로드하여 단어 시퀀스를 정수 시퀀스로 바꾸는 인코딩 작업이나 반대로 변환하는 디코딩 작업을 할 수 있습니다.

In [17]:
sp = spm.SentencePieceProcessor()
vocab_file = "imdb.model"
sp.load(vocab_file)


True

encode_as_pieces : 문장을 입력하면 서브 워드 시퀀스로 변환합니다.
encode_as_ids : 문장을 입력하면 정수 시퀀스로 변환합니다.

In [18]:
lines = [
  "I didn't at all think of it this way.",
  "I have waited a long time for someone to film"
]
for line in lines:
  print(line)
  print(sp.encode_as_pieces(line))
  print(sp.encode_as_ids(line))
  print()


I didn't at all think of it this way.
['▁I', '▁didn', "'", 't', '▁at', '▁all', '▁think', '▁of', '▁it', '▁this', '▁way', '.']
[41, 624, 4950, 4926, 139, 170, 378, 30, 58, 73, 413, 4945]

I have waited a long time for someone to film
['▁I', '▁have', '▁wa', 'ited', '▁a', '▁long', '▁time', '▁for', '▁someone', '▁to', '▁film']
[41, 142, 1364, 1121, 4, 668, 285, 93, 1079, 33, 91]



sp.GetPieceSize() 단어 집합의 크기 확인

In [19]:
sp.GetPieceSize()


5000

idToPiece : 정수로부터 맵핑되는 서브 워드로 변환합니다.

In [20]:
sp.IdToPiece(430)


'▁character'

In [21]:
sp.PieceToId('▁character')


430

In [22]:
sp.DecodeIds([41, 141, 1364, 1120, 4, 666, 285, 92, 1078, 33, 91])


'Iul wa fall aold timeooland to film'

In [23]:
sp.DecodePieces(['▁I', '▁have', '▁wa', 'ited', '▁a', '▁long', '▁time', '▁for', '▁someone', '▁to', '▁film'])


'I have waited a long time for someone to film'

In [24]:
print(sp.encode('I have waited a long time for someone to film', out_type=str))
print(sp.encode('I have waited a long time for someone to film', out_type=int))


['▁I', '▁have', '▁wa', 'ited', '▁a', '▁long', '▁time', '▁for', '▁someone', '▁to', '▁film']
[41, 142, 1364, 1121, 4, 668, 285, 93, 1079, 33, 91]
